In [1]:
from google.colab import userdata
userdata.get('MISTRAL_API_KEY')

's9WphMGMHj2Q35AhWRnUN9ylVvY7KRBM'

In [2]:
!pip install mistralai gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.9/754.9 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.38.0
    Uninstalling opentelemetry-api-1.38.0:
      Successfully uninstalled opentelemetry-api-1.38.0
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.59b0
    Uninstalling opentelemetry-semantic-conventions-0.59b0:
      Successfully uninstalled opentelemetry-semantic-conventions-0.59b0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-sdk 1.38.0 requires opent

## Mistral Agent with JSON-Schema


1.   [Custom Structured Outputs](https://docs.mistral.ai/capabilities/structured_output/custom)
2.   Listaelem



In [3]:
import os

from mistralai.client import Mistral

client = Mistral(api_key=os.environ.get("MISTRAL_API_KEY"))

inputs = [
        {
            "role": "user",
            "content": "Mi a mai legfontosabb hírek?"
        }
    ]

completion_args = {
    "temperature": 0.45,
    "max_tokens": 4096,
    "top_p": 0.2,
    "response_format": {
        "type": "json_schema",
        "json_schema": {
            "name": "response_schema",
            "schema": {
                "type": "object",
                "title": "StructuredData",
                "required": [
                    "news_title",
                    "news_content",
                    "news_topics",
                    "news_url"
                ],
                "properties": {
                    "news_url": {
                        "type": "string",
                        "format": "uri",
                        "description": "news' URL"
                    },
                    "news_title": {
                        "type": "string",
                        "description": "News' title"
                    },
                    "news_topics": {
                        "type": "string",
                        "description": "News' topics"
                    },
                    "news_content": {
                        "type": "string",
                        "description": "News' content"
                    }
                }
            }
        }
    },
}

tools = [
    {
        "toolConfiguration": None,
        "type": "web_search"
    }
]

response = client.beta.conversations.start(
    inputs=inputs,
    model="mistral-medium-latest",
    #instructions="Híradós szerkesztő vagy. Feladatod az, hogy az aktuális napi informatikai, technológiai hírek anyagját szerkezd meg, foglald össze magyarul, majd küldd el a hírcsatornáknak. Tárgyilagos és fókuszált legyél. Válaszaidban több helyről szerezd be az információkat. Híreket csoportokba szedve küldd el folyó szövegként.",
    instructions="you are J. Jonah Jameson, the loud, impatient, hard-edged editor-in-chief of the Daily Bugle. You speak in short, firm, commanding sentences. You believe in law and order, and you hate masked self-styled heroes — especially Spider-Man, who you think is a menace. You are always assertive, confrontational, and sarcastic. You never praise Spider-Man. Stay in character in every response.",
    completion_args=completion_args,
    tools=tools,
)

print(response)

SDKError: API error occurred: Status 401 Content-Type "application/json; charset=utf-8". Body: {"detail":"Unauthorized"}

## Mistral Agent SDK + Pydantic AI + Tools hasznáhasznáaa

In [4]:
import os
from mistralai.client import Mistral
from google.colab import userdata

os.environ['MISTRAL_API_KEY'] = userdata.get('MISTRAL_API_KEY')
client = Mistral(api_key=os.environ.get("MISTRAL_API_KEY"))

inputs = [
    {
        "role": "user",
        "content": "Gyűjtsd össze a mai legfontosabb híreket külön-külön kifejtve!"
    }
]

completion_args = {
    "temperature": 0.45,
    "response_format": {
        "type": "json_schema",
        "json_schema": {
            "name": "news_list_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "news_items": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "required": ["news_title", "news_content", "news_url", "news_topics"],
                            "properties": {
                                "news_title": {"type": "string", "description": "News' title"},
                                "news_content": {"type": "string", "description": "News' content"},
                                "news_url": {"type": "string", "format": "uri", "description": "news' URL"},
                                "news_topics": {"type": "string", "description": "News' topics"}
                            }
                        }
                    }
                },
                "required": ["news_items"]
            }
        }
    }
}

tools = [{"type": "web_search"}]

response = client.beta.conversations.start(
    inputs=inputs,
    model="mistral-medium-latest",
    instructions="Gyors, trendérzékeny, lényegretörő híradós hírszerkeztő vagy természetvédelem témakörben. A hírből azonnali, platformspecifikus posztot gyártasz Facebookra. A híreket egy 'news_items' nevű listába rendezd a megadott séma szerint.",
    completion_args=completion_args,
    tools=tools
)

In [5]:
import json

for entry in response.outputs:
    if entry.type == 'message.output':
        try:
            data = json.loads(entry.content)
            print(f"--- {len(data['news_items'])} hír érkezett ---\n")

            for i, item in enumerate(data['news_items'], 1):
                print(f"{i}. {item['news_title'].upper()}")
                print(f"Témák: {item['news_topics']}")
                print(f"Forrás: {item['news_url']}")
                print(f"{item['news_content']}\n" + "-"*30 + "\n")

        except (json.JSONDecodeError, KeyError) as e:
            print(f"Hiba a feldolgozás során: {e}")

--- 11 hír érkezett ---

1. A GSMA FŐIGAZGATÓJÁNAK FIGYELMEZTETÉSE EURÓPÁNAK AZ 5G-HÁLÓZATOK KIÉPÍTÉSE KAPCSÁN
Témák: 5G-hálózatok, MI-csúcstalálkozó, technológiai beruházások
Forrás: https://hu.euronews.com/next/tech-hirek
A GSMA főigazgatója, Vivek Badrinath az Euronewsnak beszélt arról, mire számíthatunk a technológiai konferencián, és kemény figyelmeztetést intézett Európának az 5G-hálózatok kiépítése kapcsán. A világ politikai és technológiai vezetői ezen a héten Indiában gyűlnek össze az éves globális MI-csúcstalálkozóra, amelynek célja az egységes mesterségesintelligencia-kormányzási keret és a nemzetközi együttműködés alapjainak lefektetése. A technológiai óriások mindent az MI-fejlesztésre tesznek fel: az idei tőkeberuházási tervük meghaladja a 700 milliárd dollárt (590 milliárd euró), ami mintegy 75%-os növekedés 2025-höz képest.
------------------------------

2. ELON MUSK SAJÁT CHIPGYÁRRAL ERŐSÍTENÉ MEG CÉGEI TECHNOLÓGIAI FÜGGETLENSÉGÉT
Témák: Elon Musk, chipgyár, technológ

[Gradio UI framework](https://www.gradio.app/guides/quickstart)

In [6]:
import gradio as gr
import json

def get_tech_news():
    try:
        # A korábbi cellában definiált logika meghívása
        inputs = [
            {
                "role": "user",
                "content": "Gyűjtsd össze a mai legfontosabb híreket külön-külön kifejtve!"
            }
        ]

        response = client.beta.conversations.start(
            inputs=inputs,
            model="mistral-medium-latest",
            instructions=" Szigorú, szkeptikus, adatorientált hírszerkeztő vagy természetvédelem témakörben. Nem írsz homályos megfogalmazásokat. A híreket egy 'news_items' nevű listába rendezd a megadott séma szerint.",
            completion_args=completion_args,
            tools=tools
        )

        output_html = ""
        for entry in response.outputs:
            if entry.type == 'message.output':
                data = json.loads(entry.content)
                for item in data['news_items']:
                    output_html += f"<div style='border: 1px solid #ddd; padding: 10px; margin-bottom: 10px; border-radius: 5px;'>"
                    output_html += f"<h3>{item['news_title']}</h3>"
                    output_html += f"<p><b>Témák:</b> {item['news_topics']}</p>"
                    output_html += f"<p>{item['news_content']}</p>"
                    output_html += f"<a href='{item['news_url']}' target='_blank'>Forrás megnyitása</a>"
                    output_html += "</div>"
        return output_html
    except Exception as e:
        return f"Hiba történt: {str(e)}"

with gr.Blocks() as demo:
    gr.Markdown("# 📰 Technológiai Híradó")
    gr.Markdown("Kattints a gombra a legfrissebb hírek letöltéséhez a Mistral AI segítségével.")
    btn = gr.Button("Hírek lekérése")
    output = gr.HTML(label="Hírek")

    btn.click(fn=get_tech_news, outputs=output)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://97824e1b14f062d570.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://97824e1b14f062d570.gradio.live


### Projekt Összefoglaló

A munkafüzet célja egy olyan automatizált technológiai híradó rendszer kidolgozása volt, amely a Mistral AI API-t és a Gradio keretrendszert ötvözi. A folyamat során beállítottuk a szükséges SDK-kat, definiáltunk egy szigorú JSON-sémát a strukturált adatkinyeréshez, és integráltuk a Mistral `web_search` eszközét, hogy a modell valós idejű információk alapján dolgozhasson.

Az eredmény egy funkcionális Gradio webes felület, amely gombnyomásra összegyűjti, rendszerezi és formázott HTML-listaként megjeleníti a legfrissebb informatikai és technológiai híreket. A rendszer képessé vált a nyers AI-válaszok strukturált feldolgozására, biztosítva a forráshivatkozások és a témakörök szerinti kategorizálás pontosságát.

### Részletes Projekt Összefoglaló

A fejlesztés elsődleges célkitűzése egy olyan intelligens hírfeldolgozó munkafolyamat kialakítása volt, amely képes a technológiai szektor legfrissebb eseményeit valós időben monitorozni és strukturált formában tálalni. A rendszer alapját a Mistral AI `web_search` képessége adja, amely lehetővé teszi, hogy a modell ne csak a statikus tudására támaszkodjon, hanem az interneten éppen elérhető aktuális forrásokat is becsatornázza a válaszadásba.

A tervezési fázis során kiemelt szerepet kapott a **Mistral AI Studio** grafikus felülete, ahol a promptok finomhangolása és az ügynök (agent) viselkedésének prototipizálása történt. Itt kísérleteztük ki a megfelelő JSON-sémát is, amely szigorú keretek közé szorítja a kimenetet, biztosítva, hogy minden hír rendelkezzen validált URL-lel, kategóriával és tömör összefoglalóval, így az adatok további feldolgozásra vagy automatikus publikálásra is alkalmassá váltak.

A projekt zárásaként a logikát egy Python környezetbe integráltuk a Mistral SDK segítségével, és egy interaktív Gradio felhasználói felületet építettünk köré. Ez a demó alkalmazás lehetővé teszi a végfelhasználók számára, hogy egyetlen gombnyomással generáljanak platformspecifikus bejegyzéseket vagy szakmai híradókat, bizonyítva az MI-alapú automatizáció hatékonyságát a tartalomgyártásban és a hírszerkesztésben.